# PhasePhyto Apple Overlap Fixes (Calibration + Rebalanced Retrain)

Fixes the two failure modes observed in `PhasePhyto_Apple_Overlap_Colab.ipynb`:

1. **Healthy bias from PV class imbalance.** PV training distribution is
   ~65% healthy / 25% scab / 11% rust. Under PP2021 the baseline predicts
   "healthy" for 37% of actual scab and 9% of actual rust.
2. **Rust collapses into scab** (43% of actual rust on PP2021).

This notebook applies two interventions on top of the existing baseline:

- **Fix A: post-hoc logit adjustment (no retrain).** Subtract `log(prior)`
  from the baseline checkpoint's logits at inference time. Implements the
  "balanced softmax" / Menon et al. 2021 logit-adjustment recipe. No target
  labels touched. Cheap. Deployable.
- **Fix B: rebalanced retrain.** Same architecture and recipe as the baseline
  but with `data.balanced_sampler: true` (`WeightedRandomSampler` over the
  three PV apple classes). Attacks the prior bias at the source. Uses the
  committed `configs/apple_overlap_plantdoc_rebalanced.yaml`.

> **If you edit any cell, click `Runtime -> Restart runtime` and `Run all`.**
> All real logic lives in the canonical scripts/configs in the repo. The
> notebook is a thin driver — change the script or the config, push the
> branch in `CONFIG['repo_branch']`, then re-run.

Prerequisite: you have already run `PhasePhyto_Apple_Overlap_Colab.ipynb`
to completion, which produced:

- `MyDrive/PhasePhyto/data/archives/apple_strict.tar`
- `MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc/best_model.pt`
- `MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc/eval_pp2021.json`
- `MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc/eval_plantdoc.json`


In [ ]:
# ============================================================
# CONFIGURATION -- edit only if you want different paths/behavior
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "repo_url": "https://github.com/kautilyaa/PhasePhyto.git",
    "repo_branch": "new_updation",
    "repo_dir": "/content/PhasePhyto",
    # Where the baseline lives (produced by the apple-overlap notebook).
    "baseline_checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc",
    # Where Fix B will write its rebalanced checkpoint and evals.
    "rebalanced_checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc_rebalanced",
    # Comparison artifacts go alongside the baseline so all three sit together.
    "comparison_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_fixes_comparison",
    "run_fix_a_calibration": True,
    "run_fix_b_retrain": False,
    "run_fix_b_eval": False,
    "run_comparison": True,
    # Optional: estimate target prior from PP2021 labels (oracle, for analysis
    # only). When False, Fix A uses the standard balanced-softmax shift
    # (uniform target prior) which needs no target labels.
    "use_oracle_target_prior": False,
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + CLONE/INSTALL REPO + DEFINE PATHS
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(CONFIG["repo_dir"])
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", CONFIG["repo_branch"],
        CONFIG["repo_url"], str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", CONFIG["repo_branch"], "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DRIVE_PROJECT_DIR = Path(CONFIG["drive_project_dir"])
ARCHIVE_DIR = DRIVE_PROJECT_DIR / "data" / "archives"
OVERLAP_ARCHIVE = ARCHIVE_DIR / "apple_strict.tar"
LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_OVERLAP_PARENT = LOCAL_DATA_ROOT / "overlap"
LOCAL_OVERLAP_ROOT = LOCAL_OVERLAP_PARENT / "apple_strict"

BASELINE_CKPT_DIR = Path(CONFIG["baseline_checkpoint_dir"])
REBALANCED_CKPT_DIR = Path(CONFIG["rebalanced_checkpoint_dir"])
COMPARISON_DIR = Path(CONFIG["comparison_dir"])

for path in [DRIVE_PROJECT_DIR, ARCHIVE_DIR, LOCAL_DATA_ROOT, COMPARISON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo ready:", REPO_DIR)
print("Overlap archive:", OVERLAP_ARCHIVE)
print("Baseline ckpt dir:", BASELINE_CKPT_DIR)
print("Rebalanced ckpt dir:", REBALANCED_CKPT_DIR)
print("Comparison dir:", COMPARISON_DIR)


In [ ]:
# ============================================================
# HELPERS
# ============================================================
# Mirror of the apple-overlap notebook's helper layout: import canonical
# script logic, define small notebook-only utilities, and fail fast if the
# committed configs aren't in the cloned branch.

import json
import shutil
import sys
import tarfile
from pathlib import Path

SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

CONFIGS_DIR = REPO_DIR / "configs"
PLANTDOC_CFG = CONFIGS_DIR / "apple_overlap_plantdoc.yaml"
PP2021_CFG = CONFIGS_DIR / "apple_overlap_pp2021.yaml"
REBALANCED_CFG = CONFIGS_DIR / "apple_overlap_plantdoc_rebalanced.yaml"
for cfg in (PLANTDOC_CFG, PP2021_CFG, REBALANCED_CFG):
    if not cfg.exists():
        raise FileNotFoundError(
            f"Missing committed config: {cfg}. Make sure CONFIG['repo_branch'] "
            f"({CONFIG['repo_branch']}) contains the apple-overlap configs."
        )


def run(cmd, *, capture: bool = False):
    print("\n$ " + " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        capture_output=capture,
    )
    if capture and result.stdout:
        print(result.stdout)
    if capture and result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): " + " ".join(map(str, cmd))
        )
    return result


def extract_tar_to_parent(archive_path: Path, parent_dir: Path, *, clean_target: bool = True) -> Path:
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    with tarfile.open(archive_path) as tf:
        top_levels = [m.name.split("/", 1)[0] for m in tf.getmembers() if m.name]
        top_level = next((n for n in top_levels if n and n != "."), None)
        if top_level is None:
            raise RuntimeError(f"Could not determine top-level folder inside {archive_path}")
    target_dir = parent_dir / top_level
    if clean_target and target_dir.exists():
        shutil.rmtree(target_dir)
    parent_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as tf:
        tf.extractall(parent_dir)
    print(f"Extracted {archive_path} -> {target_dir}")
    return target_dir


## Preflight: hydrate baseline data + check baseline checkpoint


In [ ]:
# ============================================================
# HYDRATE OVERLAP DATA TO SSD + VERIFY BASELINE ARTIFACTS
# ============================================================

if not OVERLAP_ARCHIVE.exists():
    raise FileNotFoundError(
        f"Overlap archive missing: {OVERLAP_ARCHIVE}. "
        "Run PhasePhyto_Apple_Overlap_Colab.ipynb first to produce it."
    )

if not LOCAL_OVERLAP_ROOT.exists() or not (LOCAL_OVERLAP_ROOT / "plantvillage").exists():
    if LOCAL_OVERLAP_PARENT.exists():
        shutil.rmtree(LOCAL_OVERLAP_PARENT)
    extract_tar_to_parent(OVERLAP_ARCHIVE, LOCAL_OVERLAP_PARENT, clean_target=False)
    print("Hydrated overlap archive to SSD:", LOCAL_OVERLAP_ROOT)
else:
    print("Overlap data already on SSD:", LOCAL_OVERLAP_ROOT)

baseline_ckpt = BASELINE_CKPT_DIR / "best_model.pt"
if not baseline_ckpt.exists():
    raise FileNotFoundError(
        f"Baseline checkpoint not found: {baseline_ckpt}. "
        "Run PhasePhyto_Apple_Overlap_Colab.ipynb with run_train=True first."
    )
baseline_eval_pp2021 = BASELINE_CKPT_DIR / "eval_pp2021.json"
baseline_eval_plantdoc = BASELINE_CKPT_DIR / "eval_plantdoc.json"
print(f"Baseline checkpoint: {baseline_ckpt} ({baseline_ckpt.stat().st_size / 1e6:.1f} MB)")
print(f"Baseline PP2021 eval: {'present' if baseline_eval_pp2021.exists() else 'MISSING'}")
print(f"Baseline PD eval:     {'present' if baseline_eval_plantdoc.exists() else 'MISSING'}")


## Fix A: post-hoc logit adjustment on PP2021 (no retrain)

Standard "balanced softmax" / logit-adjustment recipe (Menon et al. 2021):
at inference time, subtract `log(p_train_c)` from the c-th logit, which
counter-shifts the model toward a uniform posterior. This rebalances the
healthy-biased predictions without touching the model weights.

Two variants supported:

- **Standard:** target prior assumed uniform. No target labels needed.
- **Oracle:** target prior estimated from PP2021 labels. Upper-bound
  comparison only; do not deploy.


In [ ]:
# ============================================================
# FIX A: BUILD MODEL FROM BASELINE CHECKPOINT, RUN INFERENCE ON PP2021,
# APPLY LOGIT ADJUSTMENT, RECOMPUTE METRICS
# ============================================================

import math
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader

from phasephyto.data.datasets import PlantDiseaseDataset
from phasephyto.data.transforms import get_val_transforms
from phasephyto.models import PhasePhyto
from phasephyto.utils.config import load_config


def collect_logits(model, loader, device):
    """Collect (logits, labels) over a loader.

    Handles both 2-tuple (rgb, label) and DualTransform 3-tuple
    (rgb, clahe, label) batches, mirroring phasephyto.evaluation.domain_shift.
    """
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                rgb, clahe, targets = batch
                rgb = rgb.to(device, non_blocking=True)
                clahe = clahe.to(device, non_blocking=True)
            else:
                rgb, targets = batch
                rgb = rgb.to(device, non_blocking=True)
                clahe = None
            outputs = model(rgb, x_clahe=clahe)
            if isinstance(outputs, dict):
                logits = outputs["logits"]
            elif isinstance(outputs, (tuple, list)):
                logits = outputs[0]
            else:
                logits = outputs
            all_logits.append(logits.detach().cpu())
            all_targets.append(targets)
    return torch.cat(all_logits).numpy(), torch.cat(all_targets).numpy()


def metrics_from_preds(targets, preds, class_names):
    return {
        "accuracy": float(accuracy_score(targets, preds)),
        "f1_macro": float(f1_score(targets, preds, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(targets, preds, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(targets, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(targets, preds, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(targets, preds, labels=list(range(len(class_names)))).tolist(),
        "report": classification_report(
            targets, preds,
            labels=list(range(len(class_names))),
            target_names=class_names,
            zero_division=0,
        ),
    }


if CONFIG["run_fix_a_calibration"]:
    cfg = load_config(str(PLANTDOC_CFG))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    val_tf = get_val_transforms(cfg.data.image_size)

    pv_root = LOCAL_OVERLAP_ROOT / "plantvillage"
    pp_root = LOCAL_OVERLAP_ROOT / "plant_pathology_2021"

    # Source ImageFolder fixes the canonical class_to_idx; target uses the same.
    source_ds = PlantDiseaseDataset(root=pv_root, transform=val_tf)
    target_ds = PlantDiseaseDataset(root=pp_root, transform=val_tf, class_to_idx=source_ds.class_to_idx)
    class_names = source_ds.classes
    num_classes = source_ds.num_classes

    print(f"Classes (canonical order): {class_names}")
    print(f"PV samples: {len(source_ds)} | PP2021 samples: {len(target_ds)}")

    pv_loader = DataLoader(source_ds, batch_size=cfg.training.batch_size,
                           num_workers=cfg.data.num_workers, pin_memory=True)
    pp_loader = DataLoader(target_ds, batch_size=cfg.training.batch_size,
                           num_workers=cfg.data.num_workers, pin_memory=True)

    model = PhasePhyto(
        num_classes=num_classes,
        backbone_name=cfg.model.backbone_name,
        fusion_dim=cfg.model.fusion_dim,
        pc_scales=cfg.model.pc_scales,
        pc_orientations=cfg.model.pc_orientations,
        image_size=(cfg.data.image_size, cfg.data.image_size),
        num_heads=cfg.model.num_heads,
        dropout=cfg.model.dropout,
    ).to(device)

    state = torch.load(baseline_ckpt, map_location=device, weights_only=True)
    model.load_state_dict(state["model_state_dict"])
    print(f"Loaded baseline checkpoint (epoch {state.get('epoch', '?')}).")

    print("Collecting PV logits (for prior + sanity-check source metrics)...")
    pv_logits, pv_targets = collect_logits(model, pv_loader, device)
    print("Collecting PP2021 logits (target)...")
    pp_logits, pp_targets = collect_logits(model, pp_loader, device)

    # Source class prior pi_c from PV training distribution (use ImageFolder counts).
    source_counts = np.bincount(pv_targets, minlength=num_classes).astype(float)
    source_prior = source_counts / source_counts.sum()
    log_source_prior = np.log(np.clip(source_prior, 1e-12, None))

    if CONFIG["use_oracle_target_prior"]:
        target_counts = np.bincount(pp_targets, minlength=num_classes).astype(float)
        target_prior = target_counts / target_counts.sum()
        prior_label = "oracle (PP2021 label distribution)"
    else:
        target_prior = np.full(num_classes, 1.0 / num_classes)
        prior_label = "uniform (no target labels used)"
    log_target_prior = np.log(np.clip(target_prior, 1e-12, None))

    # logit_adjusted = logit + log(pi_target) - log(pi_source)
    shift = log_target_prior - log_source_prior
    print(f"\nSource prior: {dict(zip(class_names, source_prior.round(3)))}")
    print(f"Target prior ({prior_label}): {dict(zip(class_names, target_prior.round(3)))}")
    print(f"Logit shift (target - source, log-space): {dict(zip(class_names, shift.round(3)))}")

    pp_preds_baseline = pp_logits.argmax(axis=1)
    pp_preds_calibrated = (pp_logits + shift).argmax(axis=1)
    pv_preds_calibrated = (pv_logits + shift).argmax(axis=1)

    fix_a_results = {
        "source": metrics_from_preds(pv_targets, pv_preds_calibrated, class_names),
        "target": metrics_from_preds(pp_targets, pp_preds_calibrated, class_names),
        "delta": {},
        "use_case": "plant_disease",
        "calibration": {
            "method": "logit_adjustment",
            "target_prior_source": prior_label,
            "source_prior": source_prior.tolist(),
            "target_prior": target_prior.tolist(),
            "logit_shift": shift.tolist(),
        },
    }
    fix_a_results["delta"]["accuracy_drop"] = (
        fix_a_results["target"]["accuracy"] - fix_a_results["source"]["accuracy"]
    )
    fix_a_results["delta"]["f1_drop"] = (
        fix_a_results["target"]["f1_macro"] - fix_a_results["source"]["f1_macro"]
    )

    out_path = BASELINE_CKPT_DIR / "eval_pp2021_calibrated.json"
    out_path.write_text(json.dumps(fix_a_results, indent=2, default=str) + "\n")
    print(f"\nFix A calibrated PP2021 eval written to: {out_path}")
    print("\nBaseline vs Fix A on PP2021:")
    print(f"  accuracy:  baseline={accuracy_score(pp_targets, pp_preds_baseline):.4f}"
          f"  -> fix_a={fix_a_results['target']['accuracy']:.4f}")
    print(f"  f1_macro:  baseline={f1_score(pp_targets, pp_preds_baseline, average='macro', zero_division=0):.4f}"
          f"  -> fix_a={fix_a_results['target']['f1_macro']:.4f}")
    print("\nPer-class report (Fix A on PP2021):")
    print(fix_a_results["target"]["report"])
else:
    print("Set CONFIG['run_fix_a_calibration'] = True to run Fix A.")


## Fix B: rebalanced re-train

Re-runs the same recipe with `data.balanced_sampler: true` (committed in
`configs/apple_overlap_plantdoc_rebalanced.yaml`). Writes a new checkpoint
under `apple_overlap_plantdoc_rebalanced/` so the baseline stays intact.

Default off because it needs a GPU and ~30-45 min on a T4.


In [ ]:
# ============================================================
# FIX B: REBALANCED RE-TRAIN VIA THE COMMITTED CONFIG
# ============================================================

REBALANCED_CKPT_DIR.mkdir(parents=True, exist_ok=True)

if CONFIG["run_fix_b_retrain"]:
    run([
        sys.executable, "-m", "phasephyto.train",
        "--config", str(REBALANCED_CFG),
        "--override",
        f"checkpoint_dir={REBALANCED_CKPT_DIR}",
        f"data.root={LOCAL_OVERLAP_ROOT}",
        f"data.source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
        f"data.eval_source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.eval_target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
    ], capture=True)
else:
    print("Set CONFIG['run_fix_b_retrain'] = True to launch the rebalanced re-train.")


In [ ]:
# ============================================================
# FIX B: EVAL THE REBALANCED CHECKPOINT ON PD AND PP2021
# ============================================================

rebalanced_ckpt = REBALANCED_CKPT_DIR / "best_model.pt"

if CONFIG["run_fix_b_eval"]:
    if not rebalanced_ckpt.exists():
        raise FileNotFoundError(
            f"Rebalanced checkpoint missing: {rebalanced_ckpt}. "
            "Set CONFIG['run_fix_b_retrain']=True and rerun the previous cell first."
        )
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(REBALANCED_CFG),
        "--checkpoint", str(rebalanced_ckpt),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plantdoc"),
        "--output", str(REBALANCED_CKPT_DIR / "eval_plantdoc.json"),
    ], capture=True)
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(PP2021_CFG),
        "--checkpoint", str(rebalanced_ckpt),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plant_pathology_2021"),
        "--output", str(REBALANCED_CKPT_DIR / "eval_pp2021.json"),
    ], capture=True)
else:
    print("Set CONFIG['run_fix_b_eval'] = True to evaluate the rebalanced checkpoint.")


## Comparison: baseline vs Fix A vs Fix B


In [ ]:
# ============================================================
# THREE-WAY COMPARISON ON PP2021 (and PlantDoc when available)
# ============================================================

import pandas as pd

def load_eval(path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def per_class_rows(label, eval_payload, class_names):
    if eval_payload is None:
        return []
    cm = np.array(eval_payload["target"]["confusion_matrix"])
    rows = []
    for i, cls in enumerate(class_names):
        support = cm[i].sum()
        tp = cm[i, i]
        recall = tp / support if support else 0.0
        col_sum = cm[:, i].sum()
        precision = tp / col_sum if col_sum else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "variant": label,
            "class": cls,
            "support": int(support),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
        })
    return rows


def macro_row(label, eval_payload):
    if eval_payload is None:
        return None
    return {
        "variant": label,
        "target_accuracy": round(eval_payload["target"]["accuracy"], 4),
        "target_f1_macro": round(eval_payload["target"]["f1_macro"], 4),
        "target_precision_macro": round(eval_payload["target"]["precision_macro"], 4),
        "target_recall_macro": round(eval_payload["target"]["recall_macro"], 4),
    }


if CONFIG["run_comparison"]:
    eval_paths = {
        "baseline":       BASELINE_CKPT_DIR  / "eval_pp2021.json",
        "fix_a_calib":    BASELINE_CKPT_DIR  / "eval_pp2021_calibrated.json",
        "fix_b_rebalance": REBALANCED_CKPT_DIR / "eval_pp2021.json",
    }
    payloads = {label: load_eval(p) for label, p in eval_paths.items()}
    available = {l: p for l, p in payloads.items() if p is not None}
    if not available:
        print("No evaluation files available yet. Re-run Fix A and/or Fix B first.")
    else:
        any_payload = next(iter(available.values()))
        class_names = list(any_payload.get("source", {}).get("report", "").splitlines()[2:5]) or [
            "Apple___Apple_scab", "Apple___Cedar_apple_rust", "Apple___healthy",
        ]
        # Recover canonical class names from the confusion matrix dimensionality.
        cm_dim = len(any_payload["target"]["confusion_matrix"])
        class_names = [
            "Apple___Apple_scab", "Apple___Cedar_apple_rust", "Apple___healthy",
        ][:cm_dim]

        macro_rows = [r for r in (macro_row(l, p) for l, p in available.items()) if r]
        macro_df = pd.DataFrame(macro_rows)
        per_class_df = pd.DataFrame(
            sum((per_class_rows(l, p, class_names) for l, p in available.items()), [])
        )
        wide = per_class_df.pivot(index="class", columns="variant",
                                  values=["precision", "recall", "f1"]) if not per_class_df.empty else pd.DataFrame()

        print("\n=== PP2021 macro comparison ===")
        print(macro_df.to_string(index=False))
        print("\n=== PP2021 per-class precision/recall/F1 ===")
        if not wide.empty:
            print(wide.round(4).to_string())

        COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
        macro_df.to_csv(COMPARISON_DIR / "pp2021_macro_comparison.csv", index=False)
        per_class_df.to_csv(COMPARISON_DIR / "pp2021_per_class_comparison.csv", index=False)
        (COMPARISON_DIR / "comparison_summary.json").write_text(
            json.dumps({"macro": macro_rows, "available_variants": list(available)}, indent=2) + "\n"
        )
        print(f"\nSaved comparison artifacts under: {COMPARISON_DIR}")
else:
    print("Set CONFIG['run_comparison'] = True to build the comparison table.")


## Caveats and next steps

- **Fix A is post-hoc calibration only.** It cannot recover the rust->scab
  feature confusion (those errors happen at the feature level; logit shift
  only re-weights the decision boundary). Expect F1 to improve on
  `Apple_scab`/`Cedar_apple_rust` mostly via recall, with some precision
  loss on `Apple___healthy`.
- **Fix B should attack both axes.** Balanced sampling reduces the prior
  bias and exposes the model to ~3x more rust gradients per epoch, which
  should help the rust feature representation too. If rust recall on
  PP2021 still doesn't move past ~0.6, the residual gap is genuine
  feature shift -- next step would be a transductive intervention
  (TENT / a small PD/PP2021 fine-tune).
- **Single seed.** Multi-seed (43, 44) on Fix B would put error bars on
  any improvement before claiming it.
- **Use oracle prior with care.** `use_oracle_target_prior=True` peeks at
  PP2021 labels; it is useful as an upper bound but is not a deployable
  configuration.
